# 09.07 · Multi-agent systems

**Chapter:** Tool Use & Agents · **Milestone:** M4

## Goals
- [ ] Build a research system: lead/planner agent → parallel search sub-agents → writer
- [ ] Implement **handoffs** (one agent transfers control to a specialist)
- [ ] Measure the cost multiplier and decide when multi-agent is worth it

## Resources
- [Anthropic – How we built our multi-agent research system](https://www.anthropic.com/engineering/multi-agent-research-system) — free
- [Cognition – Don't build multi-agents](https://cognition.ai/blog/dont-build-multi-agents) — free (the counter-argument; read both)
- [OpenAI Agents SDK – Handoffs](https://openai.github.io/openai-agents-python/handoffs/) — free

Multi-agent helps when work **parallelises** and each sub-task benefits from its **own clean context** (research, broad search).
It hurts when sub-agents need shared context to make consistent decisions (e.g. writing one codebase). And it multiplies tokens.

In [ ]:
import asyncio
import json
import time

from openai import AsyncOpenAI
from pydantic import BaseModel

from aieng import rag
from aieng.llm import client

c, MODEL = client()
EXTRA = {"reasoning_effort": "none"} if c.base_url.host == "localhost" else {}
ac = AsyncOpenAI(base_url=str(c.base_url), api_key=c.api_key)
USAGE = {"calls": 0, "tokens": 0}


async def call(messages, **kw):
    r = await ac.chat.completions.create(model=MODEL, messages=messages, temperature=0, max_tokens=kw.pop("max_tokens", 800), **EXTRA, **kw)
    USAGE["calls"] += 1; USAGE["tokens"] += r.usage.total_tokens
    return r

## 1. A search sub-agent
Each sub-agent gets one sub-question and its own context window, searches (tool loop from 09.02, compact), and returns a short **condensed**
finding — not raw passages. Condensing is the point: the lead agent's context stays small.

In [ ]:
SEARCH_TOOL = [{"type": "function", "function": {"name": "search_docs", "description": "Search the encyclopedia.",
                "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}}]


async def search_agent(subquestion, max_steps=3):
    messages = [{"role": "system", "content": "Research the question with search_docs (1-3 searches). Then reply with <=60 words of findings, citing ids like [12]."},
                {"role": "user", "content": subquestion}]
    for step in range(max_steps):
        msg = (await call(messages, tools=SEARCH_TOOL, tool_choice="required" if step == 0 else "auto")).choices[0].message
        if not msg.tool_calls and step == 0:  # Ollama ignores tool_choice: nudge and retry instead
            messages.append({"role": "user", "content": "Use the tools before answering."})
            continue
        if not msg.tool_calls:
            return msg.content
        messages.append(msg.model_dump(exclude_none=True))
        for tc in msg.tool_calls:
            q = json.loads(tc.function.arguments).get("query", subquestion)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": "\n".join(f"[{p}] {t[:400]}" for p, _, t in rag.search(q, 3))})
    return (await call(messages + [{"role": "user", "content": "Summarise your findings now."}])).choices[0].message.content

## 2. Lead agent: plan → fan out → synthesise

In [ ]:
class Plan(BaseModel):
    subquestions: list[str]


async def research(question):
    plan = (await ac.chat.completions.parse(model=MODEL, temperature=0, max_tokens=400, response_format=Plan, **EXTRA, messages=[{"role": "user", "content":
            f"Split this research question into 2-4 independent sub-questions that can be searched separately:\n{question}"}])).choices[0].message.parsed
    USAGE["calls"] += 1
    print("plan:", plan.subquestions)
    findings = await asyncio.gather(*(search_agent(q) for q in plan.subquestions))  # parallel sub-agents
    notes = "\n\n".join(f"## {q}\n{f}" for q, f in zip(plan.subquestions, findings))
    final = await call([{"role": "user", "content": f"Write a 120-word answer to: {question}\nUse only these notes and keep their citations.\n\n{notes}"}])
    return final.choices[0].message.content


QUESTION = "Compare Abraham Lincoln and Ulysses S. Grant: their roles in the Civil War and what happened to each after it."
USAGE.update(calls=0, tokens=0); start = time.perf_counter()
answer = await research(QUESTION)
multi = dict(USAGE, secs=round(time.perf_counter() - start, 1))
print(answer, "\n", multi)

## 3. Baseline: one agent, same tools

In [ ]:
async def single_agent(question, max_steps=6):
    messages = [{"role": "system", "content": "Research with search_docs, then write a 120-word answer citing ids like [12]."},
                {"role": "user", "content": question}]
    for step in range(max_steps):
        msg = (await call(messages, tools=SEARCH_TOOL, tool_choice="required" if step == 0 else "auto")).choices[0].message
        if not msg.tool_calls and step == 0:  # Ollama ignores tool_choice: nudge and retry instead
            messages.append({"role": "user", "content": "Use the tools before answering."})
            continue
        if not msg.tool_calls:
            return msg.content
        messages.append(msg.model_dump(exclude_none=True))
        for tc in msg.tool_calls:
            q = json.loads(tc.function.arguments).get("query", question)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": "\n".join(f"[{p}] {t[:400]}" for p, _, t in rag.search(q, 3))})
    return "step limit"


USAGE.update(calls=0, tokens=0); start = time.perf_counter()
single = await single_agent(QUESTION)
single_usage = dict(USAGE, secs=round(time.perf_counter() - start, 1))
print(single, "\n", single_usage)
print(f"\nmulti-agent used {multi['tokens'] / max(single_usage['tokens'], 1):.1f}x the tokens of the single agent")

Anthropic reported multi-agent research using ~15× the tokens of a chat — worth it only when the task's value justifies it.
Compare answer *quality* yourself (or with an LLM judge, chapter 10) before concluding anything.

## 4. Handoffs (OpenAI Agents SDK)
A triage agent hands the conversation to a specialist, who then owns it. Different pattern from fan-out: sequential, one agent active at a time.

In [ ]:
from agents import Agent, ModelSettings, OpenAIChatCompletionsModel, Runner, set_tracing_disabled

set_tracing_disabled(True)
model = OpenAIChatCompletionsModel(model=MODEL, openai_client=ac)
settings = ModelSettings(temperature=0, extra_body=EXTRA)
billing = Agent(name="Billing agent", instructions="You handle billing questions. Start your reply with 'BILLING:'.", model=model, model_settings=settings)
tech = Agent(name="Tech agent", instructions="You handle technical problems. Start your reply with 'TECH:'.", model=model, model_settings=settings)
triage = Agent(name="Triage", instructions="Hand off billing questions to the Billing agent and technical problems to the Tech agent.",
               handoffs=[billing, tech], model=model, model_settings=settings)

for q in ["I was charged twice this month", "The app crashes when I upload a PDF"]:
    r = await Runner.run(triage, q)
    print(f"{q!r} -> handled by {r.last_agent.name}: {r.final_output[:100]!r}")

## Exercises
1. Judge both answers (multi vs single) with an LLM rubric (coverage, correctness, citations) — on 5 different research questions.
2. Add a "citation checker" agent that verifies each cited id actually supports its sentence (`rag.corpus()[id]`). How many citations fail?
3. Limit the lead agent's sub-agents by budget: if total tokens exceed N, stop spawning and synthesise what you have.

In [ ]:
# your code here